In [5]:
import import_ipynb
import data_analysis_classification

ขนาดชุดข้อมูล: Train (8101, 27), Test (2026, 27)
ฟีเจอร์ที่ผ่านเกณฑ์โหวต (>=3): ['Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Total_Revolving_Bal', 'Contacts_Count_12_mon', 'Total_Trans_Amt', 'Months_Inactive_12_mon', 'Total_Relationship_Count']


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

# Plain NumPy arrays, 1D labels (avoids Series/2D shape problems everywhere below)
X_train = np.asarray(data_analysis_classification.X_scaled, dtype=float)
X_test  = np.asarray(data_analysis_classification.X_test_scaled, dtype=float)
y_train = np.asarray(data_analysis_classification.y_train).ravel().astype(int)
y_test  = np.asarray(data_analysis_classification.y_test).ravel().astype(int)

print(X_train.shape, X_test.shape, "positive rate (train/test):", y_train.mean().round(3), y_test.mean().round(3))

(8101, 27) (2026, 27) positive rate (train/test): 0.161 0.16


In [7]:
class CatBoostClassifier:
    def __init__(self, learning_rate=0.1, n_estimators=100, max_depth=3,
                 l2_leaf_reg=3.0, border_count=32, cat_features=None,
                 prior_weight=1.0, random_state=0):
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.l2_leaf_reg = l2_leaf_reg
        self.border_count = border_count
        self.cat_features = cat_features
        self.prior_weight = prior_weight
        self.random_state = random_state

    # ---------------- helpers ----------------
    @staticmethod
    def _softmax(z):
        z = z - z.max(axis=1, keepdims=True)          # numerically stable
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def _as_array(self, X):
        X = np.asarray(X)
        if self._cats:                                 # mixed types -> object copy
            X = np.array(X, dtype=object, copy=True)
        else:
            X = X.astype(float)
        return X

    def _ordered_ts(self, codes, target, perm, prior):
        """TS for each row from rows BEFORE it in `perm` only."""
        cp, tp = codes[perm], target[perm]
        idx = np.argsort(cp, kind="stable")
        sc, st = cp[idx], tp[idx]
        n = len(sc)
        starts = np.r_[0, np.flatnonzero(np.diff(sc)) + 1]
        grp_start = np.repeat(starts, np.diff(np.r_[starts, n]))
        cs = np.cumsum(st)
        base = np.where(grp_start > 0, cs[grp_start - 1], 0.0)
        sum_before = cs - st - base
        cnt_before = np.arange(n) - grp_start
        ts_sorted = (sum_before + self.prior_weight * prior) / (cnt_before + self.prior_weight)
        tmp = np.empty(n)
        tmp[idx] = ts_sorted
        out = np.empty(n)
        out[perm] = tmp
        return out

    def _matrix(self, X, train):
        """Numeric matrix: numeric cols as-is, each cat col -> K TS columns."""
        n, d = X.shape
        cols = []
        for j in range(d):
            if j not in self._cats:
                cols.append(X[:, j].astype(float))
            elif train:
                cols.extend(self._train_ts[j])
            else:
                m = self._cat_stats[j]
                vals = X[:, j].astype(str)
                for k in range(self.K):
                    out = np.full(n, self._priors[k])
                    for i, v in enumerate(vals):
                        if v in m:
                            s, c = m[v]
                            out[i] = (s[k] + self.prior_weight * self._priors[k]) / (c + self.prior_weight)
                    cols.append(out)
        return np.column_stack(cols)

    def _quantize(self, M, fit=False):
        n, d = M.shape
        if fit:
            qs = np.linspace(0, 1, self.border_count + 1)[1:-1]
            self._borders = [np.unique(np.quantile(M[:, j], qs)) for j in range(d)]
        Xb = np.empty((n, d), dtype=np.int64)
        for j in range(d):
            Xb[:, j] = np.searchsorted(self._borders[j], M[:, j], side="left")
        return Xb

    # ---------------- one oblivious tree ----------------
    def _build_tree(self, Xb, G, H):
        n, d = Xb.shape
        K = self.K
        B = max(len(b) for b in self._borders) + 1
        leaf = np.zeros(n, dtype=np.int64)
        splits = []
        for level in range(self.max_depth):
            L = 2 ** level
            best_score, best = -np.inf, None
            for j in range(d):
                nb = len(self._borders[j]) + 1
                if nb < 2:
                    continue
                key = leaf * B + Xb[:, j]
                score = np.zeros(nb - 1)
                for k in range(K):
                    g = np.bincount(key, weights=G[:, k], minlength=L * B).reshape(L, B)
                    h = np.bincount(key, weights=H[:, k], minlength=L * B).reshape(L, B)
                    gl = np.cumsum(g, 1)[:, :nb - 1]
                    hl = np.cumsum(h, 1)[:, :nb - 1]
                    gr = g.sum(1, keepdims=True) - gl
                    hr = h.sum(1, keepdims=True) - hl
                    score += (gl**2 / (hl + self.l2_leaf_reg)
                              + gr**2 / (hr + self.l2_leaf_reg)).sum(0)
                t = int(np.argmax(score))
                if score[t] > best_score:
                    best_score, best = score[t], (j, t)
            if best is None:
                break
            splits.append(best)
            leaf = leaf * 2 + (Xb[:, best[0]] > best[1])
        L = 2 ** len(splits)
        values = np.zeros((L, K))
        for k in range(K):
            g = np.bincount(leaf, weights=G[:, k], minlength=L)
            h = np.bincount(leaf, weights=H[:, k], minlength=L)
            values[:, k] = g / (h + self.l2_leaf_reg)      # Newton step per leaf
        return splits, values, leaf

    @staticmethod
    def _leaf_index(Xb, splits):
        idx = np.zeros(len(Xb), dtype=np.int64)
        for j, t in splits:
            idx = idx * 2 + (Xb[:, j] > t)
        return idx

    # ---------------- public API ----------------
    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        self._cats = set(self.cat_features or [])
        X = self._as_array(X)
        self.classes_, yi = np.unique(np.asarray(y), return_inverse=True)
        self.K = len(self.classes_)
        n = len(yi)
        one_hot_y = np.eye(self.K)[yi]
        self._priors = one_hot_y.mean(axis=0)

        # ordered target statistics (single random permutation)
        perm = rng.permutation(n)
        self._train_ts, self._cat_stats = {}, {}
        for j in self._cats:
            uniq, codes = np.unique(X[:, j].astype(str), return_inverse=True)
            self._train_ts[j] = [self._ordered_ts(codes, one_hot_y[:, k], perm, self._priors[k])
                                 for k in range(self.K)]
            self._cat_stats[j] = {u: (one_hot_y[codes == c].sum(0), (codes == c).sum())
                                  for c, u in enumerate(uniq)}

        Xb = self._quantize(self._matrix(X, train=True), fit=True)

        self._base = np.log(np.clip(self._priors, 1e-9, None))
        predictions = np.tile(self._base, (n, 1))
        self.trees = []
        for _ in range(self.n_estimators):
            P = self._softmax(predictions)
            G = one_hot_y - P                  # residuals (negative gradient)
            H = P * (1 - P)                    # diagonal hessian
            splits, values, leaf = self._build_tree(Xb, G, H)
            predictions += self.learning_rate * values[leaf]
            self.trees.append((splits, values))
        return self

    def _raw(self, samples):
        X = self._as_array(samples)
        Xb = self._quantize(self._matrix(X, train=False))
        out = np.tile(self._base, (len(Xb), 1))
        for splits, values in self.trees:
            out += self.learning_rate * values[self._leaf_index(Xb, splits)]
        return out

    def predict_proba(self, samples):
        return self._softmax(self._raw(samples))

    def predict(self, samples):
        return self.classes_[np.argmax(self._raw(samples), axis=1)]



In [8]:
clf = CatBoostClassifier(
    n_estimators=100,
    max_depth=3
).fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Bank Churn Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))

Bank Churn Accuracy: 0.9664363277393879
Precision: 0.9508771929824561
Recall: 0.8338461538461538
F1 Score: 0.8885245901639345
ROC-AUC: 0.9900746167412834
